In [ ]:
import torch,transformers
print('torch',torch.__version__,'tf',transformers.__version__)
cap=torch.cuda.get_device_capability(0) if torch.cuda.is_available() else None
print('gpu',torch.cuda.get_device_name(0) if cap else 'none','sm',cap)
assert cap and cap[0]>=7,'dino GPU — set accelerator to GPU T4 x2 and re-run' 

In [ ]:
!pip -q install -U peft trl datasets accelerate

In [ ]:
from transformers import AutoTokenizer,AutoModelForCausalLM
B='prithivMLmods/Qwen3.5-4B-Opus-Distilled-Heretic-Thinking-Multistage-SFT-v1.0'
tok=AutoTokenizer.from_pretrained(B); tok.pad_token=tok.pad_token or tok.eos_token
model=AutoModelForCausalLM.from_pretrained(B,torch_dtype=torch.bfloat16,device_map='auto')
model.config.use_cache=False; print('loaded')

In [ ]:
import json,urllib.request
from datasets import Dataset
url="https://raw.githubusercontent.com/Kaiser0733/enilo-rehan/main/data/train_v1.jsonl"
rows=[json.loads(l) for l in urllib.request.urlopen(url).read().decode().splitlines() if l.strip()]
ds=Dataset.from_list([{'text':tok.apply_chat_template(r['messages'],tokenize=False)} for r in rows])
print('lessons',len(ds))

In [ ]:
from peft import LoraConfig,get_peft_model
pc=LoraConfig(r=16,lora_alpha=32,target_modules='all-linear',task_type='CAUSAL_LM')
model=get_peft_model(model,pc); model.print_trainable_parameters()

In [ ]:
from transformers import TrainingArguments,DataCollatorForLanguageModeling
from trl import SFTTrainer
args=TrainingArguments(output_path='/kaggle/working/run',per_device_train_batch_size=1,
 gradient_accumulation_steps=8,learning_rate=2e-4,num_train_epochs=3,
 logging_steps=5,bf16=True,save_strategy='no',report_to=[])
tr=SFTTrainer(model=model,args=args,train_dataset=ds,
 data_collator=DataCollatorForLanguageModeling(tok,mlm=False))
tr.train()
print('done')

In [ ]:
m=model.merge_and_unload()
o='/content/enilo-rehan-v1'
m.save_pretrained(o); tok.save_pretrained(o)
import os
for r,_,fs in os.walk(o):
  for f in fs: print(f,os.path.getsize(os.path.join(r,f))//1_000_000,'MB')